<a href="https://colab.research.google.com/github/geun-hyoung/python-textmining-understanding_shs/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Setting

In [ ]:
import sys
import subprocess
import platform

if "google.colab" in sys.modules:
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-y", "fonts-nanum"], check=False)
    path = "/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf"
elif platform.system() == "Windows":
    path = "C:/Windows/Fonts/malgun.ttf"
else:
    path = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"

print(f"Using font path: {path}")

import re
from string import punctuation

import warnings

import numpy as np
import pandas as pd

from collections import Counter, defaultdict
from sklearn.cluster import SpectralClustering
from sklearn.decomposition import PCA, LatentDirichletAllocation, NMF, TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer, TfidfVectorizer
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import MaxAbsScaler, MinMaxScaler, Normalizer, RobustScaler, normalize

from matplotlib.ticker import MaxNLocator, FuncFormatter
from matplotlib import pyplot, patches
import matplotlib.pyplot as plt
from PIL import Image
import seaborn as sns

import networkx as nx

import nltk
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import TreebankWordTokenizer, word_tokenize

import spacy
from spacy.lang.en import English
from spacy.lang.en.stop_words import STOP_WORDS

from textblob import Word

from wordcloud import WordCloud

%matplotlib inline

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()
warnings.filterwarnings('ignore')

### STEP 1


In [ ]:
# data Load
ori_df = pd.read_csv("datasets/shs_articles.csv")
print("number of doucments:", ori_df.shape)
ori_df.head()

In [ ]:
# data cleaning
ori_df.drop_duplicates(subset=['Article Title', 'Abstract'], inplace=True)
ori_df.dropna(subset=['Abstract', 'Article Title'], inplace=True)
print("after cleaning:", ori_df.shape[0])

In [ ]:
# data transformation : create corpus
def combine_columns(*kwargs):
    result = ""
    for sth in kwargs:
        if not pd.isna(sth):
            result += " " + str(sth)
    return result

trans_df = ori_df.copy()
trans_df["text"] = trans_df.apply(lambda x: combine_columns(x['Article Title'], x['Abstract'], x['Author Keywords']), axis=1)

custom_nolema = pd.read_csv("datasets/customized_rule_1(lemmatization).csv")
custom_stopwords = pd.read_csv("datasets/customized_rule_2(stopwords).csv")

# default : nltk stopwords
custom_nolema = list(custom_nolema['nolema'])
stop_words = stop_words.union(list(custom_stopwords['Stopwords']))

punctuations = list(punctuation)

In [ ]:
# data preprocessing
def preprocess_text(text):
    # 1. tokenization
    tokens = word_tokenize(text)

    # 2. check specific problem(i.e., do not lemmatize) : create custom list
    changed, filtered_tokens = [], []
    wordnet_lemmatizer = WordNetLemmatizer()

    for word in tokens:
        # 3. remove numbers
        word = re.sub(r'\d+', '', word)

        # 4. text lowercase
        temp_word = word.lower()

        # 5. lemmatization using TextBlob(performance better)
        lemma_word = Word(temp_word).lemmatize() if temp_word not in custom_nolema else temp_word

        # 5-1 : create custom
        if temp_word != lemma_word:
            changed.append({temp_word: lemma_word})

        filtered_tokens.append(lemma_word)

    filtered_tokens = [word for word in filtered_tokens if word not in stop_words and word not in punctuations]
    processed_text = ' '.join([token for token in filtered_tokens])

    return processed_text, changed

preprocessd_df = trans_df.copy()
preprocessd_df['corpus'], preprocessd_df['lemma_pairs'] = zip(*preprocessd_df['text'].apply(preprocess_text))
preprocessd_df.drop(columns=['text'], inplace=True)

In [ ]:
df = preprocessd_df.loc[:, ['corpus', 'Article Title']]

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(df['corpus'])
feature_names = tfidf_vectorizer.get_feature_names_out()

print("after preprocessing, documents:",tfidf_matrix.shape[0])
print("after preprocessing, keywords:",tfidf_matrix.shape[1])

### STEP 2

In [ ]:
# visualization function
def plot_keywords(x, y, y_label, title):
    plt.style.use('seaborn-whitegrid')
    plt.figure(figsize=(10,6))

    plt.plot(x, y,  marker='o', color = 'b')
    plt.gca().invert_xaxis()
    plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['left'].set_visible(False)
    plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
    plt.gca().set_facecolor('lightgray')
    plt.grid(color='white', linestyle='-', linewidth=1)

    k_formatter = FuncFormatter(lambda x, pos: f'K{x}')
    plt.gca().xaxis.set_major_formatter(k_formatter)

    plt.xticks(range(min(keyword_counts), max(keyword_counts)+1))

    plt.xlabel('Number of selected top keywords from each data')
    plt.ylabel(y_label)

    for i in [3, 4, 5, 6]:
        if i in x:
            index = x.index(i)
            plt.plot(i, y[index], marker='o', color='y')

    plt.title(title)
    plt.show()

keyword_counts = list(range(20, 1, -1))
tfidf_matrix_transposed = tfidf_matrix.toarray()

In [ ]:
mean_tfidf_scores = []

for count in keyword_counts:
    top_keywords = []

    for doc_idx in range(len(df)):
        tfidf_scores = tfidf_matrix[doc_idx].toarray()[0]
        top_indices = np.argsort(tfidf_scores)[-count:]
        top_keywords.extend([feature_names[idx] for idx in top_indices])
    top_keywords = list(set(top_keywords))

    keyword_indices = [np.where(np.array(feature_names) == keyword)[0][0] for keyword in top_keywords]
    keyword_tfidf_scores = tfidf_matrix[:, keyword_indices].toarray()

    mean_tfidf_score = np.mean(keyword_tfidf_scores)
    mean_tfidf_scores.append(mean_tfidf_score)

plot_keywords(keyword_counts, mean_tfidf_scores, 'Metric 1',  'Mean of the TF-IDF scores of a word-feature across data')

In [ ]:
mean_cosine_similarities = []

for num_keywords in keyword_counts:
    top_keywords = []

    for doc_idx in range(len(df)):
        tfidf_scores = tfidf_matrix[doc_idx].toarray()[0]
        top_indices = np.argsort(tfidf_scores)[-num_keywords:]
        top_keywords.extend([feature_names[idx] for idx in top_indices])

    top_keywords = list(set(top_keywords))

    keyword_indices = [np.where(np.array(feature_names) == keyword)[0][0] for keyword in top_keywords]
    keyword_matrix = tfidf_matrix[:, keyword_indices].T
    cosine_similarities = cosine_similarity(keyword_matrix)

    mean_cosine_similarity = np.mean(cosine_similarities)
    mean_cosine_similarities.append(mean_cosine_similarity)

plot_keywords(keyword_counts, mean_cosine_similarities,'Metric 2', 'Mean of the cosine similarities of a word-feature to other features')

In [ ]:
median_cosine_similarities = []

for count in keyword_counts:
    top_keywords = []

    for doc_idx in range(len(df)):
        tfidf_scores = tfidf_matrix[doc_idx].toarray()[0]
        top_indices = np.argsort(tfidf_scores)[-count:]
        top_keywords.extend([feature_names[idx] for idx in top_indices])

    top_keywords = list(set(top_keywords))

    keyword_indices = [np.where(np.array(feature_names) == keyword)[0][0] for keyword in top_keywords]
    keyword_matrix = tfidf_matrix[:, keyword_indices]

    keyword_matrix = keyword_matrix.T.toarray()

    centroid = np.mean(keyword_matrix, axis=0)
    centroid = np.asarray(centroid).reshape(1, -1)

    cosine_similarities = []

    for idx in range(keyword_matrix.shape[0]):
        word_vector = keyword_matrix[idx, :]
        word_vector = np.asarray(word_vector).reshape(1, -1)
        cosine_similarity_value = cosine_similarity(word_vector, centroid)
        cosine_similarities.append(cosine_similarity_value[0][0])


    mean_cosine_similarity = np.mean(cosine_similarities)
    median_cosine_similarities.append(mean_cosine_similarity)

plot_keywords(keyword_counts, median_cosine_similarities, "Metric 3", "Cosine similarity between a word-feature and the centroid of features : median")

In [ ]:
mean_dot_product_scores = []

for count in keyword_counts:
    top_keywords = []

    for doc_idx in range(len(df)):
        tfidf_scores = tfidf_matrix[doc_idx].toarray()[0]
        top_indices = np.argsort(tfidf_scores)[-count:]
        top_keywords.extend([feature_names[idx] for idx in top_indices])

    top_keywords = list(set(top_keywords))

    keyword_indices = [np.where(np.array(feature_names) == keyword)[0][0] for keyword in top_keywords]
    keyword_tfidf_scores = tfidf_matrix[:, keyword_indices].toarray()
    dot_product_scores = np.dot(keyword_tfidf_scores.T, keyword_tfidf_scores)

    mean_dot_product_score = np.mean(dot_product_scores)
    mean_dot_product_scores.append(mean_dot_product_score)

plot_keywords(keyword_counts, mean_dot_product_scores,"Metric 4", "Mean of the dot product scores of a word-feature to other features")

In [ ]:
mean_lda_scores = []

for count in keyword_counts:
    top_keywords = []

    for doc_idx in range(len(df)):
        tfidf_scores = tfidf_matrix[doc_idx].toarray()[0]
        top_indices = np.argsort(tfidf_scores)[-count:]
        top_keywords.extend([feature_names[idx] for idx in top_indices])

    top_keywords = list(set(top_keywords))
    keyword_indices = [np.where(np.array(feature_names) == keyword)[0][0] for keyword in top_keywords]
    keyword_matrix = tfidf_matrix[:, keyword_indices].toarray()

    num_topics = 1
    lda_model = LatentDirichletAllocation(n_components=num_topics)
    lda_model.fit(keyword_matrix)

    lda_scores = lda_model.components_

    mean_lda_score = np.mean(lda_scores)
    mean_lda_scores.append(mean_lda_score)

plot_keywords(keyword_counts, mean_lda_scores,"Metric 5", "LDA score of a word-feature for the single topic of entire dataset")

In [ ]:
metric1_values = median_cosine_similarities
metric2_values = mean_cosine_similarities
metric3_values = mean_tfidf_scores
metric4_values = mean_lda_scores
metric5_values = mean_dot_product_scores

metric1_normalized = (metric1_values - np.min(metric1_values)) / (np.max(metric1_values) - np.min(metric1_values)) * 40 + 60
metric2_normalized = (metric2_values - np.min(metric2_values)) / (np.max(metric2_values) - np.min(metric2_values)) * 45 + 55
metric3_normalized = (metric3_values - np.min(metric3_values)) / (np.max(metric3_values) - np.min(metric3_values)) * 75 + 25
metric4_normalized = (metric4_values - np.min(metric4_values)) / (np.max(metric4_values) - np.min(metric4_values)) * 75 + 25
metric5_normalized = (metric5_values - np.min(metric5_values)) / (np.max(metric5_values) - np.min(metric5_values)) * 80 + 20

plt.style.use('seaborn-whitegrid')
plt.figure(figsize=(10,6))

plt.gca().invert_xaxis()

plt.gca().spines['right'].set_visible(False)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['left'].set_visible(False)
plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
plt.gca().set_facecolor('lightgray')

plt.xticks(range(min(keyword_counts), max(keyword_counts)+1))

k_formatter = FuncFormatter(lambda x, pos: f'K{x}')

plt.gca().xaxis.set_major_formatter(k_formatter)
plt.grid(color='white', linestyle='-', linewidth=1)
plt.xlabel('Number of selected top keywords from each data')
plt.ylabel('Degree of importance (relative)')

plt.plot(keyword_counts, metric1_normalized, marker='o', label='Metric 1 Cosine similarity from other data', color='red')
plt.plot(keyword_counts, metric2_normalized, marker='s', label='Metric 2 Cosine similarity from the mean data', color='blue')
plt.plot(keyword_counts, metric3_normalized, marker='^', label='Metric 3 TF-IDF score', color='green')
plt.plot(keyword_counts, metric4_normalized, marker='*', label='Metric 4 LDA scoes', color='orange')
plt.plot(keyword_counts, metric5_normalized, marker='x', label='Metric 5 Dot product score', color='purple')

plt.xlabel('Number of selected top keywords from each data')
plt.ylabel('Degree of importance (relative)')
plt.title('Find optimal number of selected top keywords from each data')
plt.legend(frameon = True, framealpha=1)

plt.show()

In [ ]:
top_keywords_per_document = []
feature_names = tfidf_vectorizer.get_feature_names_out()

for i in range(len(df)):
    tfidf_scores = zip(feature_names, tfidf_matrix[i].toarray()[0])
    sorted_tfidf_scores = sorted(tfidf_scores,key=lambda x: x[1], reverse=True)

    top_keywords_per_document.append([keyword for keyword,_ in sorted_tfidf_scores[:4]])

all_keywords=[keyword for document in top_keywords_per_document for keyword in document]
selected_keywords=list(set(keyword for keyword in all_keywords if all_keywords.count(keyword) >=2))

vectorizer = TfidfVectorizer(vocabulary=selected_keywords, norm='l2')
tfidf_matrix = vectorizer.fit_transform(df['corpus'])
tfidf_matrix_transposed = tfidf_matrix.T.toarray()

In [ ]:
from itertools import combinations
from sklearn.preprocessing import StandardScaler

vectorizer = TfidfVectorizer(vocabulary=selected_keywords)
feature_names = vectorizer.get_feature_names_out()

tfidf_matrix = vectorizer.fit_transform(df['corpus'])
tfidf_matrix_transposed = tfidf_matrix.T.toarray()

scaler = StandardScaler()
tfidf_scaled = scaler.fit_transform(tfidf_matrix_transposed)

gmm_cluster_results = []
keyword_frequency = defaultdict(int)

# GMM based Clustering
for i in range(10):
    gmm = GaussianMixture(n_components=100, covariance_type='diag', random_state=i+60)
    gmm.fit(tfidf_scaled)
    cluster_labels = gmm.predict(tfidf_scaled)

    df_clustered_keywords = pd.DataFrame(feature_names, columns=['keyword'])
    df_clustered_keywords['cluster'] = cluster_labels

    cluster_counts = df_clustered_keywords['cluster'].value_counts()
    max_cluster_id = cluster_counts.idxmax()
    max_cluster_keywords = set(df_clustered_keywords[df_clustered_keywords['cluster'] == max_cluster_id]['keyword'].values.tolist())

    gmm_cluster_results.append(max_cluster_keywords)

    for keyword in max_cluster_keywords:
        keyword_frequency[keyword] += 1

# def jaccard_similarity(set1, set2):
#     return len(set1 & set2) / len(set1 | set2)

# jaccard_scores = []
# for s1, s2 in combinations(gmm_cluster_results, 2):
#     score = jaccard_similarity(s1, s2)
#     jaccard_scores.append(score)

# mean_jaccard = np.mean(jaccard_scores)

temp_keywords = [k for k, v in keyword_frequency.items() if v >= 9]
core_keywords = [keyword for keyword in selected_keywords if keyword not in temp_keywords]

### STEP 3

In [ ]:
core_keywords = pd.read_csv("datasets/core_keyword.csv")
core_keywords = list(core_keywords['keyword'])

vectorizer = TfidfVectorizer(vocabulary=core_keywords)
tfidf_matrix = vectorizer.fit_transform(df['corpus'])
print(tfidf_matrix.shape[0])
print(tfidf_matrix.shape[1])

In [ ]:
shs_mask = np.array(Image.open('datasets/mask.png'))

tfidf = tfidf_matrix
tfidf_array = tfidf.toarray()

tfidf_means = np.mean(tfidf_array, axis=0).reshape(-1, 1)
cosine_sim = cosine_similarity(tfidf_array.T)
cosine_means = np.mean(cosine_sim, axis=0).reshape(-1, 1)

centroid_features = np.mean(tfidf_array.T, axis=0)
cosine_similarities = cosine_similarity(tfidf_array.T, centroid_features.reshape(1, -1))

dot_product = tfidf_array.T.dot(tfidf_array)
dot_product_means = np.mean(dot_product, axis=0).reshape(-1, 1)

lda_model = LatentDirichletAllocation(n_components=1)
lda_model.fit(tfidf_array)
lda_scores = lda_model.components_[0].reshape(-1, 1)

scaler = MinMaxScaler()
scaled_values = scaler.fit_transform(np.hstack([tfidf_means, cosine_means, cosine_similarities, dot_product_means, lda_scores]))

keyword_values = np.sum(scaled_values, axis=1)

if np.isnan(keyword_values).any():
    keyword_values = np.nan_to_num(keyword_values)

palettes = ['spring', 'summer', 'seismic','PuBu']

stopwords_list = ['based','proposed','two','may','existing','low','need' ,'using','real']
stopwords_set = set(stopwords_list)

frequencies = dict(zip(vectorizer.get_feature_names_out(), keyword_values))
filtered_frequencies = {word: freq for word, freq in frequencies.items() if word not in stopwords_set and word in set(core_keywords)}

wordcloud = WordCloud(background_color ='White',
                      mode="RGB",
                      mask = shs_mask,
                      colormap='summer',
                      width = 800,
                      height = 800,
                      font_path=path,
                      scale=10, stopwords = set(stopwords_list),
                      random_state=7,
                      ).generate_from_frequencies(filtered_frequencies)

plt.axis('off')
plt.tight_layout(pad=0)
plt.imshow(wordcloud, interpolation='antialiased')
plt.savefig('results/wordcloud_plot.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# spectral clustering
metrics = ['cosine', 'euclidean']
cluster_range = range(5, 50)

results = {}

for metric in metrics:
    results[metric] = []

    if metric == 'cosine':
        affinity = 'precomputed'
        sim_matrix = cosine_similarity(tfidf_matrix)

    elif metric == 'euclidean':
        affinity = 'precomputed'
        distance_matrix = euclidean_distances(tfidf_matrix)
        sigma = np.mean(distance_matrix)
        sim_matrix = np.exp(-distance_matrix ** 2 / (2.0 * ( sigma ** 2)))

    for n_clusters in cluster_range:
        silhouette_avg_list = []

        # Spectral Clustering
        for i in range(10):
            sc = SpectralClustering(n_clusters=n_clusters, affinity = affinity, assign_labels = 'discretize', random_state = i).fit(sim_matrix)
            labels=sc.labels_

            silhouette_avg_list.append(silhouette_score(sim_matrix, labels, metric = metric))

        results[metric].append(np.mean(silhouette_avg_list))

In [ ]:
plt.figure(figsize=(10,6))

plt.gca().spines['right'].set_visible(False)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['left'].set_visible(False)

plt.gca().set_facecolor('lightgray')
plt.grid(color='white', linestyle='-', linewidth=1)

for metric in metrics:

    if metric == 'cosine':
        color = 'g'
    elif metric == 'euclidean':
        color = 'c'

    plt.plot(cluster_range, results[metric], label=f'Silhouette_Avg_{metric}', marker='o', color = color)

plt.title('Evaluation of clustering results')
plt.xlabel('Number of clusters')
plt.ylabel('Silhouette_Avg')
plt.legend()

plt.savefig('results/Silhouette_Avgt.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
n_clusters = 13
n_topics = 1
n_top_words = 10

vectorizer = TfidfVectorizer(vocabulary=core_keywords)
tfidf_matrix = vectorizer.fit_transform(df['corpus'])

sim_matrix = cosine_similarity(tfidf_matrix)

clustering = SpectralClustering(n_clusters=n_clusters, affinity='precomputed', random_state=42).fit(sim_matrix)
labels = clustering.labels_

nmf = NMF(n_components=n_topics, random_state=0)
lda = LatentDirichletAllocation(n_components=n_topics, random_state=0)

result_df = pd.DataFrame(columns=['Cluster', 'NMF_Keywords', 'LDA_Keywords'])
for i in range(n_clusters):
    cluster_data = tfidf_matrix[labels == i]

    nmf.fit(cluster_data)
    nmf_topic_words = nmf.components_.argsort()[:, ::-1]
    nmf_keywords = ", ".join([vectorizer.get_feature_names_out()[index] for index in nmf_topic_words[0, :n_top_words]])

    lda.fit(cluster_data)
    lda_topic_words = lda.components_.argsort()[:, ::-1]
    lda_keywords = ", ".join([vectorizer.get_feature_names_out()[index] for index in lda_topic_words[0, :n_top_words]])

    result_df = result_df._append({'Cluster': i, 'NMF_Keywords': nmf_keywords, 'LDA_Keywords': lda_keywords}, ignore_index=True)

writer = pd.ExcelWriter('results/result_topic.xlsx', engine='xlsxwriter')
result_df.to_excel(writer, sheet_name='Topic_Modeling', index=False)

for cluster_num in range(n_clusters):
    cluster_indices = np.where(labels == cluster_num)[0]
    cluster_sim_matrix = sim_matrix[cluster_indices, :][:, cluster_indices]

    top_docs_indices = np.argsort(-cluster_sim_matrix.sum(axis=1))[:10]
    top_docs_titles = df['Article Title'].iloc[cluster_indices[top_docs_indices]].values

    bottom_docs_indices = np.argsort(cluster_sim_matrix.sum(axis=1))[-10:]
    bottom_docs_titles = df['Article Title'].iloc[cluster_indices[bottom_docs_indices]].values

    top_bottom_docs_df = pd.DataFrame({'Top 10 Articles': top_docs_titles, 'Bottom 10 Articles': bottom_docs_titles})  # 수정됨
    top_bottom_docs_df.to_excel(writer, sheet_name=f'Cluster_{cluster_num}_articles', index=False)

writer._save()

In [ ]:
cos_sim = cosine_similarity(tfidf_matrix)
mean_sim = np.mean(cos_sim)
binary_adj_matrix = np.where(cos_sim >= mean_sim, 1, 0)

node_order = np.argsort(labels)
sorted_binary_adj_matrix = binary_adj_matrix[node_order, :][:, node_order]

def draw_adjacency_matrix(G, node_order=None, partitions=[], colors=[]):
    adjacency_matrix = nx.to_numpy_array(G, dtype=np.bool_, nodelist=node_order)

    #Plot adjacency matrix in toned-down black and white
    fig = plt.figure(figsize=(5, 5)) # Use plt.figure
    plt.imshow(adjacency_matrix, # Use plt.imshow
                  cmap="Greys",
                  interpolation="none")

    assert len(partitions) == len(colors)
    ax = plt.gca() # Use plt.gca()
    current_idx = 0
    for partition, color in zip(partitions, colors):
        ax.add_patch(patches.Rectangle((current_idx, current_idx), # Use patches.Rectangle
                                          len(partition), # Width
                                          len(partition), # Height
                                          facecolor="none",
                                          edgecolor=color,
                                          linewidth=2))
        current_idx += len(partition)
    # Save the plot in high quality
    plt.savefig('results/adjacency_matrix_plot.png', dpi=300, bbox_inches='tight')
    plt.show() # Keep plt.show() after savefig

G = nx.from_numpy_array(sorted_binary_adj_matrix)

partitions = []
for i in range(n_clusters):
    partitions.append([idx for idx, cluster in enumerate(labels) if cluster == i])

colors = ['#FFFF00' for _ in range(n_clusters)]
draw_adjacency_matrix(G, partitions=partitions, colors=colors)

### STEP 4

In [ ]:
node_names = {
    0: 'User perspective on adoption',
    1: 'Security',
    2: 'Energy Management',
    3: 'Architecture',
    4: 'Activity Recognition',
    5: 'Context Awareness',
    6: 'Healthcare',
    7: 'Service Robot',
    8: 'Distributed Computing',
    9: 'Expansion',
    10: 'Wireless Networks',
    11: 'Internet of Things',
    12: 'Elderly Care'
}

centroids = np.array([np.mean(tfidf_matrix[clustering.labels_ == i].toarray(), axis=0) for i in range(n_clusters)])
similarity_matrix = cosine_similarity(centroids)
top_clusters = {i: sorted(range(len(similarity_matrix[i])), key=similarity_matrix[i].__getitem__, reverse=True)[1:4] for i in range(n_clusters)}

G = nx.Graph()
for cluster, neighbors in top_clusters.items():
    for neighbor in neighbors:
        G.add_edge(cluster, neighbor)

G = nx.relabel_nodes(G, node_names)

In [ ]:
# create network
def draw_graph(G, layout, node_colors, node_sizes, title, save_path=None):
    fig, ax = plt.subplots(figsize=(20, 15))
    pos = layout(G, seed=0)
    nx.draw(G, pos, with_labels=False, node_size=node_sizes, node_color=[node_colors[node] for node in G], edgecolors='black', font_size=8, font_weight='bold', ax=ax)

    for node, (x, y) in pos.items():
        plt.text(x, y, node, fontsize=16, ha='center', va='center', fontweight = 'bold')

    plt.title(title)

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')

    plt.show()

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

def create_and_draw_graph(similarity_matrix, node_names, threshold_k, title, save_path=None):
    G = nx.Graph()
    n_clusters = similarity_matrix.shape[0]

    for i in range(n_clusters):
        top_k_indices = np.argsort(similarity_matrix[i])[::-1][1:threshold_k + 1]
        for neighbor in top_k_indices:
            weight = float(similarity_matrix[i, neighbor])
            G.add_edge(i, neighbor, weight=weight)

    G = nx.relabel_nodes(G, node_names)

    num_nodes = G.number_of_nodes()
    num_edges = G.number_of_edges()
    avg_degree = sum(dict(G.degree()).values()) / num_nodes if num_nodes > 0 else 0
    density = nx.density(G)

    degree_centrality = nx.degree_centrality(G)
    weighted_degree = dict(G.degree(weight='weight'))

    avg_clustering = nx.average_clustering(G, weight='weight') if num_nodes > 0 else 0.0

    communities = list(nx.algorithms.community.greedy_modularity_communities(G, weight='weight'))
    modularity = nx.algorithms.community.modularity(G, communities, weight='weight') if communities else 0.0

    print(f"\n--- Metrics for Top {threshold_k} Connections ---")
    print(f"Number of Nodes: {num_nodes}")
    print(f"Number of Edges: {num_edges}")
    print(f"Average Degree: {avg_degree:.2f}")
    print(f"Density: {density:.4f}")
    print(f"Average Clustering Coefficient: {avg_clustering:.4f}")
    print(f"Modularity: {modularity:.4f}")

    top_strength = sorted(weighted_degree.items(), key=lambda x: x[1], reverse=True)[:5]
    print("Top 5 nodes by weighted degree (strength):")
    for node, score in top_strength:
        print(f"  - {node}: {score:.4f}")

    output_dir = Path('results')
    output_dir.mkdir(parents=True, exist_ok=True)

    summary_row = pd.DataFrame([
        {
            'top_k': threshold_k,
            'num_nodes': num_nodes,
            'num_edges': num_edges,
            'avg_degree': avg_degree,
            'density': density,
            'avg_clustering': avg_clustering,
            'modularity': modularity,
        }
    ])
    summary_path = output_dir / 'network_metrics_summary.csv'
    summary_row.to_csv(summary_path, mode='a', index=False, header=not summary_path.exists(), encoding='utf-8-sig')

    node_metrics = pd.DataFrame({
        'node': list(G.nodes()),
        'degree': [G.degree(node) for node in G.nodes()],
        'weighted_degree': [weighted_degree[node] for node in G.nodes()],
        'degree_centrality': [degree_centrality[node] for node in G.nodes()],
    }).sort_values('weighted_degree', ascending=False)
    node_metrics_path = output_dir / f'network_node_metrics_top_{threshold_k}.csv'
    node_metrics.to_csv(node_metrics_path, index=False, encoding='utf-8-sig')

    node_colors = {node: '#a1d67f' for node in G.nodes()}
    sizes = [G.degree(node) * 2000 for node in G]

    draw_graph(
        G,
        lambda G, seed=None: nx.spring_layout(G, k=0.05, seed=7, weight=None),
        node_colors,
        sizes,
        title,
        save_path=save_path,
    )

    return G

thresholds = [1, 3, 5]
base_save_path = 'results/network_graph_top_{}.png'

summary_reset_path = Path('results') / 'network_metrics_summary.csv'
if summary_reset_path.exists():
    summary_reset_path.unlink()

for k in thresholds:
    title = f'Network Graph (Top {k} Connections)'
    save_path = base_save_path.format(k)
    _ = create_and_draw_graph(similarity_matrix, node_names, k, title, save_path)

In [ ]:
highlight_clusters = ['Context Awareness', 'Architecture']
sub_1 = ['User perspective on adoption', 'Activity Recognition', 'Elderly Care', 'Service Robot', 'Healthcare']
sub_2 = ['Internet of Things', 'Distributed Computing', 'Wireless Networks', 'Security', 'Expansion', 'Energy Management']

node_colors = {node: 'lightgreen' for node in G.nodes()}

for node in highlight_clusters:
    if node in node_colors:
        node_colors[node] = 'limegreen'

for node in sub_1:
    if node in node_colors:
        node_colors[node] = 'lightsalmon'

for node in sub_2:
    if node in node_colors:
        node_colors[node] = 'lightskyblue'

sizes = [G.degree(node) * 2000 for node in G]

draw_graph(G, lambda G, seed=None: nx.spring_layout(G, k=0.05, seed=seed), node_colors, sizes, 'Network Graph with Highlighted Clusters')